# Process-bigraph co-simulation vs the embedded vcell-fvsolver hybrid solver

_Investigation `cosim-vs-embedded-hybrid` — coder reproduction notebook._

**Question.** Can a process-bigraph co-simulation of a PDE process and a Smoldyn process reproduce the hybrid PDE/particle solver embedded in vcell-fvsolver to within its stochastic error, and at what computational cost?

A head-to-head comparison of a modular (process-bigraph) and a hard-coded (vcell-fvsolver) implementation of the same PDE/particle hybrid algorithm.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-pde-particle/viva-pde-particle').is_dir():
    REPO = Path('/home/runner/work/viva-pde-particle/viva-pde-particle')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_pde_particle.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Each side alone, and the particle-to-grid handoff (`handoff-units-and-binning`)

**Question.** Before coupling anything: does each half of the co-simulation behave correctly on its own,
and does the particle-to-grid handoff (nearest-node binning, boundary element volumes,
counts to µM) follow vcell-fvsolver's conventions?

**Objective.** Four checks, all on the shared CartesianGrid:
(1) FV diffusion of cos(πx/L)cos(πy/L) (21×21, L = 10 µm, D = 1, dt = 0.01, t = 1 s)
    against exp(-2π²Dt/L²);
(2) 20,000 particles diffusing from the centre cell (D = 1, 4 seeds): the variance of the
    binned positions, minus dx²/6 (the initial cell spread plus Sheppard's binning
    correction), against 2Dt;
(3) the histogram sum equals the molecule count at every record;
(4) 100,000 uniform particles after 2 s of diffusion (11×11, 4 seeds): the counts per
    node class (corner, edge, interior) match their element-volume shares.

**Hypothesis.** The FV process reproduces continuum diffusion to discretization accuracy. Smoldyn's binned
particle variance grows as 2Dt per axis. Binning conserves molecules exactly. Uniformly
distributed particles give a uniform binned concentration up to and including the
boundary nodes, because boundary elements have 1/2 (edge) or 1/4 (corner) volume as in
VCell's CartesianMesh.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `particle-diffusion-baseline` | `viva_pde_particle.composites.examples.particle_diffusion` | 0 | n_particles=20000, diffusion=1.0, start=center, dt=0.01 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.particle_diffusion`** — `spec_viva_pde_particle_composites_examples_particle_diffusion` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.particle_diffusion` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: handoff-units-and-binning ===
STUDY = 'handoff-units-and-binning'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**handoff**


In [ ]:
# handoff
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| fv-matches-continuum-diffusion | kind=derived_scalar field=a1_fv_cosine_rel_err | op < value 0.01 provenance {'kind': 'theory', 'note': 'Second order in space and first order in time; expected error ~1e-3 on this grid.'} |
| particle-variance-matches-2Dt | kind=derived_scalar field=a1_particle_variance_rel_err | op < value 0.03 provenance {'kind': 'theory', 'note': 'Free Brownian motion; 80,000 molecule samples give ~0.5% sampling error on the variance.'} |
| histogram-conserves-molecules | kind=derived_scalar field=a1_histogram_count_mismatch | op == value 0 provenance {'kind': 'theory', 'note': 'Every molecule maps to exactly one node (clamped nearest-node index).'} |
| boundary-volumes-give-uniform-concentration | kind=derived_scalar field=a1_boundary_class_max_abs_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'Binomial sampling; |z| < 4 for a correct volume convention (corner 1/4, edge 1/2).'} |


## Study: Field-to-particle coupling, A_particle -> 0 at rate k·[B](x) (`field-modulated-decay`)

**Question.** Does the Smoldyn process read the PDE field at each molecule as vcell-fvsolver does
(nearest node, value from the previous coupling step)? Does the coupled model approach
its deterministic continuum limit?

**Objective.** Part 1: B rises linearly from 0 to 1 µM across x; 4,000 immobile A per run; k = 1/(µM·s);
8 seeds; t = 1 s. Compare per-column survival to exp(-k·B_col·t) with binomial z-scores.
Part 2: B is a Gaussian (1 µM peak, D_B = 0.5); 20,000 A with D_A = 1; 8 seeds. Compare
the ensemble-mean A (µM) at t = 1 s to continuum_model() solved by FVReactionDiffusion
alone (relative L2 over the grid, and total A).

**Hypothesis.** With a static field and immobile particles, each grid column survives as exp(-k·B·t)
exactly in expectation, and nothing decays where B = 0. With diffusing A and B, the
ensemble-mean particle concentration matches the all-continuous FV solution of the same
model, to within sampling noise.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `field-modulated-decay-baseline` | `viva_pde_particle.composites.examples.field_modulated_decay` | 0 | n_particles=4000, k=1.0, d_a=0.0, d_b=0.0, b_profile=linear, dt=0.01 |

Declared parameter sets (`study.yaml` variants):

- **native-vcell-slab** — `n_particles=4000`, `k=1.0`, `b_profile=linear`, `dt=0.01`
- **continuum-limit** — `n_particles=20000`, `k=1.0`, `d_a=1.0`, `d_b=0.5`, `b_profile=gaussian`, `dt=0.01`


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.field_modulated_decay`** — `spec_viva_pde_particle_composites_examples_field_modulated_decay` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.field_modulated_decay` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: field-modulated-decay ===
STUDY = 'field-modulated-decay'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**field-modulated-decay**


In [ ]:
# field-modulated-decay
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| survival-follows-local-field | kind=derived_scalar field=a2_static_field_max_abs_z | op < value 4.0 provenance {'kind': 'theory', 'note': "Smoldyn's first-order reaction probability per step is 1-exp(-rate·dt), so survival is exp(-k·B·t) exactly in expectation."} |
| no-decay-where-field-is-zero | kind=derived_scalar field=a2_zero_field_decays | op == value 0 provenance {'kind': 'theory', 'note': 'Rate k·0 = 0 at the B = 0 nodes.'} |
| approaches-continuum-limit | kind=derived_scalar field=a2_continuum_rel_l2_err | op < value 0.06 provenance {'kind': 'theory', 'note': 'Per-node sampling noise of the 8-run mean is about 1/sqrt(8·165), roughly 2.8%; the threshold allows about 2x that.'} |
| total-matches-continuum | kind=derived_scalar field=a2_continuum_total_rel_err | op < value 0.02 provenance {'kind': 'theory', 'note': '160,000 total molecule samples give ~0.3% sampling error on the total.'} |
| native-vcell-follows-local-field | kind=derived_scalar field=a2_native_static_field_max_abs_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'Same exact expectation as the co-simulation test; validates the headless native reference path.'} |
| cosim-matches-native-static | kind=derived_scalar field=a2_cosim_vs_native_static_survival_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'Equal expected survival under both solvers; 8 runs each.'} |
| cosim-matches-native-diffusing | kind=derived_scalar field=a2_cosim_vs_native_diffusing_profile_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'Same model, mesh and coupling scheme in both solvers; differences only from sampling (8 runs each).'} |
| cosim-matches-native-total | kind=derived_scalar field=a2_cosim_vs_native_diffusing_survival_rel_diff | op < value 0.02 provenance {'kind': 'theory', 'note': '160,000 molecule samples per solver; sampling error ~0.5%.'} |


## Study: Mass balance across the particle ↔ field handoff (`two-way-exchange`)

**Question.** When molecules move both ways between particle and field representations (A_particle
→ B_field and B_field → A_particle by field-dependent creation), is total A + B
conserved and is the right steady state reached? Do both solvers agree?

**Objective.** 11×11×3 slab (10×10×1 µm); k1 = 1 s⁻¹, k2 = 0.5 s⁻¹; B₀ = 0.2 µM (12,044 molecules);
D_A = D_B = 1; dt = 0.01; t = 5 s; 16 seeds per solver. Measured:
- mass balance and steady-state ratio;
- co-sim vs native totals and profiles;
- a creation-only experiment (B a static catalyst, 2 s, 8 seeds), comparing created
  A with the exact expectation k2·[B]·V·602.214·t.

**Hypothesis.** Each side applies only its own half of each mixed reaction, so A + B is conserved in
expectation up to the O(dt) lag of the explicit coupling, and total A / total B → k2 / k1.
Field-dependent creation must use each grid node's actual cell volume (1/2–1/8 at
boundaries); otherwise creation is overproduced by Σ(full cells)/Σ(actual cells).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `two-way-exchange-slab` | `viva_pde_particle.composites.examples.two_way_exchange` | 0 | k1=1.0, k2=0.5, b0=0.2, dt=0.01 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.two_way_exchange`** — `spec_viva_pde_particle_composites_examples_two_way_exchange` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.two_way_exchange` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: two-way-exchange ===
STUDY = 'two-way-exchange'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**two-way-exchange**


In [ ]:
# two-way-exchange
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cosim-conserves-mass | kind=derived_scalar field=a3_cosim_mass_balance_max_rel_dev | op < value 0.03 provenance {'kind': 'theory', 'note': 'Conserved in expectation; the residual comes from the explicit-coupling lag plus the ~2-step creation start-up artifact.'} |
| cosim-reaches-steady-state | kind=derived_scalar field=a3_cosim_steady_ratio_rel_err | op < value 0.03 provenance {'kind': 'theory', 'note': 'Detailed balance of A ⇌ B; 16 seeds.'} |
| cosim-creation-is-exact | kind=derived_scalar field=a3_cosim_creation_ratio | op range low 0.97 high 1.02 provenance {'kind': 'theory', 'note': 'Exact expectation k2·[B]·V·602.214·t; the lower slack covers the ~2-step start-up artifact.'} |
| native-overproduction-matches-boundary-volume-defect | kind=derived_scalar field=a3_native_creation_vs_bug_factor | op range low 0.95 high 1.05 provenance {'kind': 'theory', 'note': 'Characterizes the reported defect (virtualcell/vcell-fvsolver#24). This test will fail once the defect is fixed upstream; then replace it with a mass-balance test for the native solver.'} |


## Study: A_particle + B_field → C_field: two-way nonlinear coupling, both solvers (`bimolecular-hybrid`)

**Question.** When particles consume a field (B substantially depleted by A), do the co-simulation
and the native VCell hybrid agree with each other and with the deterministic continuum
limit?

**Objective.** 11×11×3 slab; 20,000 A (D = 1, Poisson initial placement in both solvers);
B₀ = 0.5 µM (≈30,000 molecules, D = 1); C₀ = 0 (D = 1); k = 1/(µM·s); dt = 0.01;
t = 3 s; 16 seeds per solver. Compared:
- surviving fraction, produced C and final x-profiles (co-sim vs native);
- survival vs the continuum FV solution.

**Hypothesis.** The particle side removes A at rate k·[B](x), and the PDE side removes k·[A_binned]·[B]
from B and adds it to C. With 20,000 A, both solvers' ensemble means approach the
continuum solution (A treated as a field). The solvers differ only by sampling.
B + C is conserved exactly (PDE side).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `bimolecular-slab` | `viva_pde_particle.composites.examples.bimolecular_hybrid` | 0 | n_particles=20000, k=1.0, b0=0.5, dt=0.01 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.bimolecular_hybrid`** — `spec_viva_pde_particle_composites_examples_bimolecular_hybrid` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.bimolecular_hybrid` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bimolecular-hybrid ===
STUDY = 'bimolecular-hybrid'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**bimolecular-hybrid**


In [ ]:
# bimolecular-hybrid
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| solvers-agree-on-survival | kind=derived_scalar field=a4_cosim_vs_native_survival_max_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'Same model, mesh and coupling; independent ensembles of 16.'} |
| solvers-agree-on-product | kind=derived_scalar field=a4_cosim_vs_native_c_max_z | op < value 4.0 provenance {'kind': 'theory', 'note': 'C is produced on the PDE side from binned A; any timing or units mismatch shows up here first.'} |
| cosim-near-continuum | kind=derived_scalar field=a4_cosim_vs_continuum_survival_max_abs_diff | op < value 0.01 provenance {'kind': 'theory', 'note': '20,000 molecules per run; the remainder is explicit-coupling lag plus discreteness.'} |
| native-near-continuum | kind=derived_scalar field=a4_native_vs_continuum_survival_max_abs_diff | op < value 0.01 provenance {'kind': 'theory', 'note': 'As above.'} |


## Study: Coupling error vs Smoldyn step multiplier, co-simulation (two schemes) and native VCell (`coupling-interval-convergence`)

**Question.** How does the error of the lagged explicit coupling grow with the coupling interval
k·Δt (Smoldyn steps once per k PDE steps)? Does the co-simulation reproduce
fvsolver's coupling error, and does process-bigraph's plain scheduling
("start-of-interval") do better or worse?

**Objective.** A4 model with k = 2/(µM·s), 20,000 A, Δt = 0.005, t = 1.6 s, 64 seeds per
configuration, step multiplier k ∈ {1, 2, 4, 8, 16, 32}.
The error is the max over time of |mean surviving fraction − continuum limit|
(A continuous, Δt/10). The order is fitted for k ≥ 4, above the sampling floor
(~5·10⁻⁴).

**Hypothesis.** Explicit lagged coupling is first order in k·Δt. The co-sim "fvsolver" mode applies the
same lag as the embedded solver, so it has the same error. "start-of-interval" reads
the field at the start of the Smoldyn step instead of at T+(k−1)Δt, so its error
differs; its sign and size are to be measured.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `bimolecular-k2` | `viva_pde_particle.composites.examples.bimolecular_hybrid` | 0 | k=2.0, dt=0.005, step_multiplier=1 |

Declared parameter sets (`study.yaml` variants):

- **step-multiplier-sweep** — `k=2.0`, `dt=0.005`, `step_multiplier=16`


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.bimolecular_hybrid`** — `spec_viva_pde_particle_composites_examples_bimolecular_hybrid` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.bimolecular_hybrid` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: coupling-interval-convergence ===
STUDY = 'coupling-interval-convergence'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**coupling-convergence**


In [ ]:
# coupling-convergence
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cosim-fvsolver-mode-first-order | kind=derived_scalar field=a5_cosim_fvsolver_order | op range low 0.7 high 1.5 provenance {'kind': 'theory', 'note': 'Explicit lagged operator splitting is first order in the coupling interval.'} |
| native-first-order | kind=derived_scalar field=a5_native_order | op range low 0.7 high 1.5 provenance {'kind': 'theory', 'note': 'Same scheme (SimTool.cpp main loop).'} |
| cosim-reproduces-native-coupling-error | kind=derived_scalar field=a5_fvsolver_mode_vs_native_err_ratio_k16 | op range low 0.8 high 1.25 provenance {'kind': 'theory', 'note': 'Same lag semantics (verified in tests/test_scheduling_semantics.py), so the same coupling error.'} |


## Study: Wall time of the co-simulation vs the native VCell hybrid, with breakdown (`performance-scaling`)

**Question.** What does decomposing the hybrid solver into process-bigraph processes cost? How does
the co-simulation's wall time compare with the native embedded solver as the grid and
the particle count grow, and how much of it is framework overhead?

**Objective.** A4 model (A_p + B_f → C_f) on N×N×3 slabs, dt = 0.01, t = 1 s (100 steps), one process,
median of 3 runs. Sweeps:
- N ∈ {11, 21, 41, 81} with 20,000 particles;
- particles ∈ {2·10³, 2·10⁴, 2·10⁵} on 21×21×3.
Co-sim time is split into PDE update, particle update and framework (the remainder).
Native time is split into input generation, solve and read.

**Hypothesis.** The co-simulation pays a per-step framework overhead (scheduling, state updates and
data marshaling between processes) that matters most for small problems. For large
problems, PDE and particle work dominate in both implementations.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `bimolecular-perf` | `viva_pde_particle.composites.examples.bimolecular_hybrid` | 0 | n_particles=20000, dt=0.01 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_pde_particle.composites.examples.bimolecular_hybrid`** — `spec_viva_pde_particle_composites_examples_bimolecular_hybrid` (a plain, editable dict)


_composite spec file for `viva_pde_particle.composites.examples.bimolecular_hybrid` not found under `viva_pde_particle/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: performance-scaling ===
STUDY = 'performance-scaling'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**performance**


In [ ]:
# performance
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| framework-overhead-bounded | kind=derived_scalar field=a6_cosim_framework_ms_per_step_max | op < value 10.0 provenance {'kind': 'theory', 'note': 'A two-process composite should cost no more than a few milliseconds per step to schedule and marshal.'} |
| cosim-not-slower-than-native-at-scale | kind=derived_scalar field=a6_grid_n81_cosim_over_native_solve | op < value 2.0 provenance {'kind': 'theory', 'note': 'The modular decomposition should not cost more than 2× the embedded solver at scale.'} |
